# Decision Transformer

Chen et al. 2021, "Decision Transformer: Reinforcement Learning via Sequence Modeling" ([arXiv:2106.01345](https://arxiv.org/abs/2106.01345)).

Return-conditioned causal sequence modeling over interleaved (return-to-go, state, action) tokens. **Honesty note**: trained here on the real NGSIM traffic field reinterpreted as an offline-imitation control dataset (one spatial bin = one trajectory, action = observed speed change, reward = closeness to a free-flow-speed target) -- see `model.py`'s docstring for the full data-adaptation note. The transformer mechanism itself is exactly as the paper defines it.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from transformer_playground.data import load_ngsim_traffic_field
from transformer_playground.device import resolve_device
from model import DecisionTransformerModel, build_control_trajectories

device = resolve_device('auto')
print('device:', device)

In [ ]:
field = load_ngsim_traffic_field(space_bins=64, time_bins=200)
context_len = 10
states, actions, rtg, valid_starts = build_control_trajectories(field, context_len=context_len)

S = states.shape[1]
s_train_end = int(0.8 * S)
train_starts = [(t0, s) for (t0, s) in valid_starts if s < s_train_end]
test_starts = [(t0, s) for (t0, s) in valid_starts if s >= s_train_end]
print(f'{len(train_starts)} train windows, {len(test_starts)} test windows, context_len={context_len}')

def make_batch(starts, batch_size):
    idx = torch.randint(0, len(starts), (batch_size,))
    S_list, A_list, R_list, Y_list = [], [], [], []
    for i in idx:
        t0, s = starts[i]
        S_list.append(states[t0:t0+context_len, s, :])
        A_list.append(actions[t0:t0+context_len, s].unsqueeze(-1))
        R_list.append(rtg[t0:t0+context_len, s].unsqueeze(-1))
        Y_list.append(actions[t0:t0+context_len, s].unsqueeze(-1))
    return (torch.stack(R_list).to(device), torch.stack(S_list).to(device),
            torch.stack(A_list).to(device), torch.stack(Y_list).to(device))

In [ ]:
model = DecisionTransformerModel(state_dim=2, action_dim=1, d_model=64, n_heads=4, n_layers=3, d_ff=256, max_context_len=context_len).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()

history = {'train_loss': []}
batch_size = 32
for step in range(300):
    R, S_, A, Y = make_batch(train_starts, batch_size)
    pred = model(R, S_, A)
    loss = loss_fn(pred, Y)
    opt.zero_grad(); loss.backward(); opt.step()
    history['train_loss'].append(loss.item())
    if step % 50 == 0:
        print(f'step {step:4d} | train_loss {loss.item():.4f}')

In [ ]:
plt.plot(history['train_loss'])
plt.xlabel('step')
plt.ylabel('train action MSE')
plt.title('Decision Transformer training loss, NGSIM-derived control trajectories')
plt.show()